# FORA Model + Internal Features Merge
Bu notebook `dummy_internal_features.csv` (sol) ile `FORA_MODEL.xlsx` (sağ) dosyalarını birleştirir,
ardından türetilmiş (derived) feature'ları hesaplar.

In [1]:
import pandas as pd
import numpy as np

# 1) Verileri oku
df_model = pd.read_excel('FORA_MODEL.xlsx')
df_internal = pd.read_csv('dummy_internal_features.csv')

print(f'FORA_MODEL shape: {df_model.shape}')
print(f'Internal features shape: {df_internal.shape}')

FORA_MODEL shape: (81, 190)
Internal features shape: (81, 15)


In [2]:
# 2) Left join: CSV (sol) <- FORA_MODEL (sağ)
df_model_merge = df_model.rename(columns={'Min Tarih': 'min_tarih', 'Max Tarih': 'max_tarih'})
df_model_merge = df_model_merge.drop(columns=['max_tarih'])

df_merged = df_internal.merge(df_model_merge, on='min_tarih', how='left')
print(f'Merged shape: {df_merged.shape}')

Merged shape: (81, 203)


In [3]:
# 3) Derived Features

# Bileşik / Basit dönüşümler
df_merged["TLREF_Bilesik"] = ((1 + df_merged["TLREF"] / 100 / 365) ** 365 - 1) * 100
df_merged["TCMB_1M_Basit"] = ((1 + df_merged["TCMB 1M"] / 100) ** (1 / 365) - 1) * 36500
df_merged["TCMB_3M_Basit"] = ((1 + df_merged["TCMB 3M"] / 100) ** (1 / 12) - 1) * 1200

# Exponential spread
df_merged["EXP(CBavg-TLREF)"] = np.exp(df_merged["TCMB_3M_Basit"] - df_merged["TLREF"])

# Welcome annual spreads
df_merged["w/TLREF"] = df_merged["osawelcomeannual"] - df_merged["TLREF_Bilesik"]
df_merged["w/1month"] = df_merged["osawelcomeannual"] - df_merged["TCMB 1M"]
df_merged["w/3month"] = df_merged["osawelcomeannual"] - df_merged["TCMB 3M"]

# Welcome annual w/current (delta) spreads
df_merged["w/TLREF(deltas)"] = df_merged["osawelcomeannualwcurrent"] - df_merged["TLREF_Bilesik"]
df_merged["w/1month(deltas)"] = df_merged["osawelcomeannualwcurrent"] - df_merged["TCMB 1M"]
df_merged["w/3month(deltas)"] = df_merged["osawelcomeannualwcurrent"] - df_merged["TCMB 3M"]

# TCMB basit vs TLREF spreads
df_merged["<1month-TLREF"] = df_merged["TCMB_1M_Basit"] - df_merged["TLREF"]
df_merged["<3month-TLREF"] = df_merged["TCMB_3M_Basit"] - df_merged["TLREF"]

print('Derived features eklendi!')
print(f'Final shape: {df_merged.shape}')

Derived features eklendi!
Final shape: (81, 214)


In [4]:
# 4) Derived feature'ları kontrol et
derived_cols = [
    'TLREF_Bilesik', 'TCMB_1M_Basit', 'TCMB_3M_Basit', 'EXP(CBavg-TLREF)',
    'w/TLREF', 'w/1month', 'w/3month',
    'w/TLREF(deltas)', 'w/1month(deltas)', 'w/3month(deltas)',
    '<1month-TLREF', '<3month-TLREF'
]
df_merged[derived_cols].head(10)

,TLREF_Bilesik,TCMB_1M_Basit,TCMB_3M_Basit,EXP(CBavg-TLREF),w/TLREF,w/1month,w/3month,w/TLREF(deltas),w/1month(deltas),w/3month(deltas),<1month-TLREF,<3month-TLREF
0,61.169530,43.210292,48.049863,1.336404,9.410470,16.57,10.40,-2.989530,4.17,-2.00,-4.549588,0.289983
1,62.620709,43.683730,47.828887,0.436681,5.939291,13.82,8.72,-3.500709,4.38,-0.72,-4.973710,-0.828553
2,64.194153,43.748411,47.607480,0.133432,2.845847,12.20,7.54,-2.394153,6.96,2.30,-5.873229,-2.014160
3,64.584835,43.722544,47.776830,0.124582,6.035165,15.82,10.86,-3.024835,6.76,1.80,-6.137076,-2.082790
4,62.602521,43.502401,47.405232,0.289093,4.767479,12.91,8.18,-2.782521,5.36,0.63,-5.143839,-1.241008
5,63.312977,43.683730,47.698700,0.250549,4.967023,13.54,8.64,-2.832977,5.74,0.84,-5.399070,-1.384100
6,62.650002,43.515364,47.587924,0.337041,8.149998,16.32,11.33,-3.540002,4.63,-0.36,-5.160111,-1.087551
7,64.422001,43.897020,47.268020,0.082705,6.377999,15.73,11.82,-5.672001,3.68,-0.23,-5.863480,-2.492480
8,63.597790,43.450533,47.287632,0.139506,5.692210,14.91,10.28,-3.747790,5.47,0.84,-5.806747,-1.969648
9,63.531407,43.838895,47.261482,0.141542,5.998593,14.55,10.56,-4.121407,4.43,0.44,-5.377745,-1.955158


In [5]:
# 5) Kaydet
output_path = 'FORA_MODEL_MERGED.xlsx'
df_merged.to_excel(output_path, index=False)
print(f'Kaydedildi: {output_path}')
print(f'Final shape: {df_merged.shape}')

Kaydedildi: FORA_MODEL_MERGED.xlsx
Final shape: (81, 214)
